# Apply this code to sample TS from model.

In [1]:
import os

# Set to '0' for the first GPU, '1' for the second, etc.
# Or use a comma-separated string for multiple visible devices, e.g., '0,1'
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # UPDATE

# Verify the setting (optional)
print(f"CUDA_VISIBLE_DEVICES is set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

# Now, when you import your deep learning library, it will only see the specified device(s)
import torch
print(f"PyTorch sees {torch.cuda.device_count()} CUDA device(s).")
if torch.cuda.is_available():
    print(f"Current PyTorch device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

# For TensorFlow
# import tensorflow as tf
# print(f"TensorFlow sees {len(tf.config.list_physical_devices('GPU'))} GPU device(s).")

CUDA_VISIBLE_DEVICES is set to: 0
PyTorch sees 1 CUDA device(s).
Current PyTorch device: 0
Device name: NVIDIA A100-PCIE-40GB


In [2]:
# --- Importing and defining some functions ----
import os
import torch
import py3Dmol
import numpy as np

from typing import Optional
from torch import tensor
from e3nn import o3
from torch_scatter import scatter_mean

from oa_reactdiff.model import LEFTNet

default_float = torch.float64
torch.set_default_dtype(default_float)  # Use double precision for more accurate testing


def remove_mean_batch(
    x: tensor, 
    indices: Optional[tensor] = None
) -> tensor:
    """Remove the mean from each batch in x

    Args:
        x (tensor): input tensor.
        indices (Optional[tensor], optional): batch indices. Defaults to None.

    Returns:
        tensor: output tensor with batch mean as 0.
    """
    if indices == None:
         return x - torch.mean(x, dim=0)
    mean = scatter_mean(x, indices, dim=0)
    x = x - mean[indices]
    return x


def draw_in_3dmol(mol: str, fmt: str = "xyz") -> py3Dmol.view:
    """Draw the molecule

    Args:
        mol (str): str content of molecule.
        fmt (str, optional): format. Defaults to "xyz".

    Returns:
        py3Dmol.view: output viewer
    """
    viewer = py3Dmol.view(1024, 576)
    viewer.addModel(mol, fmt)
    viewer.setStyle({'stick': {}, "sphere": {"radius": 0.36}})
    viewer.zoomTo()
    return viewer


def assemble_xyz(z: list, pos: tensor) -> str:
    """Assembling atomic numbers and positions into xyz format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    natoms =len(z)
    xyz = f"{natoms}\n\n"
    for _z, _pos in zip(z, pos): #.numpy()):
        xyz += f"{_z}\t" + "\t".join([str(x) for x in _pos]) + "\n"
    return xyz

### Building a LEFTNet model

A simple test is performed to verify SE(3) symmetry. The model here is for testing, so we only need to build a very small model.

Note: [LEFTNet](https://arxiv.org/abs/2304.04757) is a new SOTA-level SE(3) graph neural network. Although we use LEFTNet here, the properties it exhibits are model-independent (other SE(3) models, such as [EGNN](https://arxiv.org/pdf/2102.09844.pdf), will give the same results)

TL: EGNN is not SE$(3)$  equivariant?

In [3]:
num_layers = 2
hidden_channels = 8
in_hidden_channels = 4
num_radial = 4

model =  LEFTNet(
    num_layers=num_layers,
    hidden_channels=hidden_channels,
    in_hidden_channels=in_hidden_channels,
    num_radial=num_radial,
    object_aware=False,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)

/misc/home/guest50/micromamba/envs/oa_reactdiff_m/lib/python3.10/site-packages/torch_geometric/nn/conv/message_passing.py:972: UserWarning: 'EquiMessage.jittable' is deprecated and a no-op. Please remove its usage.
  warnings.warn(f"'{self.__class__.__name__}.jittable' is deprecated "


7882



### Create an "Object-Aware" LEFTNet

In [4]:
# --- Importing necessary function ---
from torch.utils.data import DataLoader

from oa_reactdiff.trainer.pl_trainer import DDPMModule


from oa_reactdiff.dataset import ProcessedTS1x, ProcessedSCAN
from oa_reactdiff.diffusion._schedule import DiffSchedule, PredefinedNoiseSchedule

from oa_reactdiff.diffusion._normalizer import FEATURE_MAPPING
from oa_reactdiff.analyze.rmsd import batch_rmsd

from oa_reactdiff.utils.sampling_tools import assemble_sample_inputs, write_tmp_xyz

In [5]:
!pwd

/home/guest50/OAReactDiff



### Import the pre-trained model and redefine the schedule.

In [6]:
# TL fix: {
from oa_reactdiff.trainer.pl_trainer import DDPMModule
# } fix. Why didn' this carry over from the previous cell import statement?

device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda")
print(device) # TL
print(device.index)
print(device.type)


tspath = os.path.abspath(os.path.join(os.getcwd(), "oa_reactdiff","trainer"))
print(tspath)
# zenodo_pretrained_ckpt
ddpm_trainer = DDPMModule.load_from_checkpoint(
    checkpoint_path="./oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/leftnet-SCAN-6-w_selfoops-lr2.5e-4-rcmconly_passerini-03516f3022c5/ddpm-epoch=1899-val-totloss=736.23.ckpt",    
    #checkpoint_path="./oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/leftnet-SCAN-6-w_selfoops-lr2.5e-4-rcmconly_passerini-03516f3022c5/ddpm-epoch=1799-val-totloss=787.66.ckpt", --
    #
    #checkpoint_path="./pretrained-ts1x-diff.ckpt", # original
    #checkpoint_path=f"{tspath}/checkpoint/OAReactDiff/leftnet-0-f1ff7dc18fa3/ddpm-epoch=1999-val-totloss=509.31.ckpt", # Our recapitulation
    map_location=device,
)
ddpm_trainer = ddpm_trainer.to(device)

cuda
None
cuda
/misc/home/guest50/OAReactDiff/oa_reactdiff/trainer


/misc/home/guest50/micromamba/envs/oa_reactdiff_m/lib/python3.10/site-packages/torch_geometric/nn/conv/message_passing.py:972: UserWarning: 'EquiMessage.jittable' is deprecated and a no-op. Please remove its usage.
  warnings.warn(f"'{self.__class__.__name__}.jittable' is deprecated "


In [7]:
noise_schedule: str = "polynomial_2"
timesteps: int = 150
precision: float = 1e-5

gamma_module = PredefinedNoiseSchedule(
            noise_schedule=noise_schedule,
            timesteps=timesteps,
            precision=precision,
        )
schedule = DiffSchedule(
    gamma_module=gamma_module,
    norm_values=ddpm_trainer.ddpm.norm_values
)
ddpm_trainer.ddpm.schedule = schedule
ddpm_trainer.ddpm.T = timesteps
ddpm_trainer = ddpm_trainer.to(device)

In [8]:
def prep_ddpm_trainer(ckpt_path: str, device=device):
    ddpm_trainer = DDPMModule.load_from_checkpoint(
        checkpoint_path=ckpt_path,
        map_location=device,
    )
    ddpm_trainer = ddpm_trainer.to(device)

    noise_schedule: str = "polynomial_2"
    timesteps: int = 150
    precision: float = 1e-5
    
    gamma_module = PredefinedNoiseSchedule(
                noise_schedule=noise_schedule,
                timesteps=timesteps,
                precision=precision,
            )
    schedule = DiffSchedule(
        gamma_module=gamma_module,
        norm_values=ddpm_trainer.ddpm.norm_values
    )
    ddpm_trainer.ddpm.schedule = schedule
    ddpm_trainer.ddpm.T = timesteps
    ddpm_trainer = ddpm_trainer.to(device)
    return ddpm_trainer


### Prepare dataset and data loader and select a reaction involving multiple molecules

In [9]:
import pickle

# npz_path = "./oa_reactdiff/data/C6H6_5710-0-wSL-wBL-w3pBC/train.pkl" # w  self-loops after Angstrom correction
# 24/06/2026: update to check filtering results
npz_path = "./oa_reactdiff/data/C6H6_5710-filtered/train.pkl"

train_pkl = pickle.load(open(npz_path, "rb"))

In [10]:
npz_path_valid = "./oa_reactdiff/data/C6H6_5710-filtered/valid_addprop.pkl"
valid_pkl = pickle.load(open(npz_path_valid, "rb"))

In [11]:
npz_path_test = "/scr/trond/SCAN/C6H6_5710-filtered-test.pkl"
test_pkl = pickle.load(open(npz_path_test, "rb"))

In [12]:
pos = np.array(test_pkl['transition_state']['positions'][0])

In [13]:
z_num = test_pkl['transition_state']['charges'][0]

In [14]:
num2sym = {1: "H", 6: "C"}

In [15]:
test_xyz = assemble_xyz([num2sym[num] for num in z_num], pos)
print(test_xyz)

12

C	-0.516900648343	-0.174294636273	-1.067970800786
C	-1.8274168923979999	0.281855646809	-0.665555170387
C	0.7204072213550001	0.3062477260559999	-0.429570871693
C	-1.5569540082030002	-0.259379067575	0.501820927983
C	0.746055826415	0.301589139332	0.917287820757
C	-0.49997865703499994	-0.314533123402	1.535236083393
H	-2.717388734537	0.6617476993339999	-1.181950702773
H	1.5862774498539998	0.587394946715	-1.041990254096
H	-0.312462548756	-1.355347111461	1.8643291315180002
H	1.5520855533639997	0.7188507795929999	1.5318461719469998
H	-0.842730787701	0.22958547243399996	2.4369996781390006
H	-0.42212630402	-0.9837174715619998	-1.81221449399



In [16]:
test_pkl['transition_state']['rxn'][0]

'C6H6_5710-TS12 CON(2,6)'

In [17]:
train_pkl["use_ind"][:10]

[0, 1, 2, 3, 5, 6, 7, 8, 9, 10]

In [18]:
valid_pkl["use_ind"][:10]

[4, 16, 114, 116, 119, 127, 143, 154, 160, 190]

In [19]:
test_pkl["use_ind"][:10]

[34, 64, 98, 130, 146, 161, 189, 197, 200, 202]

In [20]:
#assert sorted(list(set(train_pkl2["use_ind"] + valid_pkl2["use_ind"]))) == list(range(len(train_pkl2["transition_state"]["rxn"])))

In [21]:
train_pkl["reactant"]["rxn"][177]

'C6H6_5710-TS446 CON(114,174)'

In [22]:
len(test_pkl["use_ind"])

331

In [23]:
train_pkl["reactant"]['num_atoms'][177]

12

In [24]:
selected_passerini_rcmconly_pts = [13605, 17605, 1206, 7394, 15965, 5435, 17328]
selected_strecker_rcmconly_pts = [3978, 5528, 7950]
selected_wl2_pts = [6029, 784, 11193]

In [25]:
from parse import parse
from pprint import pprint

In [26]:
set(selected_passerini_rcmconly_pts+selected_strecker_rcmconly_pts+selected_wl2_pts)

{784,
 1206,
 3978,
 5435,
 5528,
 6029,
 7394,
 7950,
 11193,
 13605,
 15965,
 17328,
 17605}

In [27]:
prefixes = {'ALD1',
 'EN1',
 'HDF1',
 'WL1',
 'f260_DFG1',
 'rcmconly_passerini',
 'rcmconly_strecker'}

select_prefixes = {'WL1', 'rcmconly_passerini', 'rcmconly_strecker'}
rxn_pattern = "{prefix}-TS{id_num}"

for i, rxn in enumerate(test_pkl["transition_state"]["rxn"]): # valid, not train
    parsed = parse(rxn_pattern, rxn)
    prefix = parsed["prefix"]
    id_num = parsed["id_num"]
    #prefixes.add(prefix)
    if prefix not in select_prefixes: 
        continue
    if int(id_num) in set(selected_passerini_rcmconly_pts+selected_strecker_rcmconly_pts+selected_wl2_pts):
        print(rxn)
        print(i)
        print(i in test_pkl["use_ind"]) # valid, not train
        print("")

In [28]:
select_rxns = []

selection = {'WL1': selected_wl2_pts, 'rcmconly_strecker': selected_strecker_rcmconly_pts, 'rcmconly_passerini': selected_passerini_rcmconly_pts}

for prefix, id_list in selection.items():
    for id_num in id_list:
        select_rxns.append(f"{prefix}-TS{id_num}")

print(select_rxns)

['WL1-TS6029', 'WL1-TS784', 'WL1-TS11193', 'rcmconly_strecker-TS3978', 'rcmconly_strecker-TS5528', 'rcmconly_strecker-TS7950', 'rcmconly_passerini-TS13605', 'rcmconly_passerini-TS17605', 'rcmconly_passerini-TS1206', 'rcmconly_passerini-TS7394', 'rcmconly_passerini-TS15965', 'rcmconly_passerini-TS5435', 'rcmconly_passerini-TS17328']


In [29]:
# assert sorted(list(set(train_pkl["use_ind"] + valid_pkl["use_ind"]))) == list(range(len(train_pkl["transition_state"]["rxn"])))

In [30]:
len(train_pkl["transition_state"]["rxn"])

6603

In [31]:
len(test_pkl["transition_state"]["rxn"])

6603

In [32]:
train_pkl["use_ind"] == test_pkl["use_ind"]

False

In [33]:
selected_train_indices = []
selected_test_indices = []
selected_rxns_dict = {}
for i, rxn in enumerate(train_pkl["transition_state"]["rxn"]):
    if rxn in select_rxns:
        print(rxn)
        print(i)
        print(i in train_pkl["use_ind"])
        if i in train_pkl["use_ind"]:
            selected_train_indices.append(train_pkl["use_ind"].index(i))
        elif i in test_pkl["use_ind"]:
            selected_test_indices.append(test_pkl["use_ind"].index(i))
        selected_rxns_dict[i] = rxn
        print("")

In [34]:
selected_train_indices

[]

In [35]:
selected_test_indices

[]

In [36]:
# update, just selecting some random validation TSes for C6H6 for now
import random

random.seed(33)
#selected_valid_pre_indices = sorted(list(random.sample(valid_pkl["use_ind"], 10)))
#selected_valid_pre_indices = sorted(list(random.sample(valid_pkl["use_ind"], 100)))
selected_test_pre_indices = sorted(list(random.sample(test_pkl["use_ind"], 100)))

selected_test_indices = []
for i in selected_test_pre_indices:
    selected_test_indices.append(test_pkl["use_ind"].index(i))
print(selected_test_indices)

[1, 3, 4, 16, 26, 28, 29, 32, 33, 35, 36, 41, 47, 56, 60, 61, 62, 63, 76, 80, 85, 92, 94, 95, 97, 98, 103, 109, 113, 119, 124, 125, 126, 138, 139, 141, 143, 144, 145, 154, 155, 157, 158, 162, 163, 164, 170, 178, 179, 181, 183, 185, 186, 187, 194, 198, 199, 201, 209, 211, 215, 218, 220, 223, 227, 232, 234, 239, 242, 245, 246, 248, 254, 255, 257, 259, 261, 262, 265, 268, 270, 272, 273, 279, 283, 284, 289, 292, 295, 300, 301, 304, 305, 315, 316, 317, 319, 323, 326, 328]


In [37]:
# valid dataset to test against:
dataset = ProcessedSCAN(
    npz_path=npz_path_test, #valid,
    center=True,
    pad_fragments=0,
    device=device,
    zero_charge=False,
    remove_h=False,
    single_frag_only=False,
    swapping_react_prod=False,
    use_by_ind=True,
    keep_use_by_ind_ordered=True, # Used for analysis but not training? 18/07/2025
)
loader = DataLoader(
    dataset, 
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=dataset.collate_fn
)
itl = iter(loader)
idx = -1 # TL: why?

len(dataset)

331

In [38]:
len(test_pkl["use_ind"])

331

In [39]:
dataset.raw_dataset["use_ind"] == test_pkl["use_ind"]

True

In [40]:
# TL visualize each state:
atomic_num2sym = {
    1: 'H',    2: 'He',   3: 'Li',   4: 'Be',   5: 'B',    6: 'C',    7: 'N',    8: 'O',    9: 'F',    10: 'Ne',
    11: 'Na',  12: 'Mg',  13: 'Al',  14: 'Si',  15: 'P',   16: 'S',   17: 'Cl',  18: 'Ar',  19: 'K',   20: 'Ca',
    21: 'Sc',  22: 'Ti',  23: 'V',   24: 'Cr',  25: 'Mn',  26: 'Fe',  27: 'Co',  28: 'Ni',  29: 'Cu',  30: 'Zn',
    31: 'Ga',  32: 'Ge',  33: 'As',  34: 'Se',  35: 'Br',  36: 'Kr',  37: 'Rb',  38: 'Sr',  39: 'Y',   40: 'Zr',
    41: 'Nb',  42: 'Mo',  43: 'Tc',  44: 'Ru',  45: 'Rh',  46: 'Pd',  47: 'Ag',  48: 'Cd',  49: 'In',  50: 'Sn',
    51: 'Sb',  52: 'Te',  53: 'I',   54: 'Xe',  55: 'Cs',  56: 'Ba',  57: 'La',  58: 'Ce',  59: 'Pr',  60: 'Nd',
    61: 'Pm',  62: 'Sm',  63: 'Eu',  64: 'Gd', 65: 'Tb',  66: 'Dy',  67: 'Ho',  68: 'Er',  69: 'Tm',  70: 'Yb',
    71: 'Lu',  72: 'Hf',  73: 'Ta',  74: 'W',   75: 'Re',  76: 'Os',  77: 'Ir',  78: 'Pt',  79: 'Au',  80: 'Hg',
    81: 'Tl',  82: 'Pb',  83: 'Bi',  84: 'Po',  85: 'At',  86: 'Rn',  87: 'Fr',  88: 'Ra',  89: 'Ac',  90: 'Th',
    91: 'Pa',  92: 'U',   93: 'Np',  94: 'Pu',  95: 'Am',  96: 'Cm',  97: 'Bk',  98: 'Cf',  99: 'Es', 100: 'Fm',
    101: 'Md', 102: 'No', 103: 'Lr', 104: 'Rf', 105: 'Db', 106: 'Sg', 107: 'Bh', 108: 'Hs', 109: 'Mt', 110: 'Ds',
    111: 'Rg', 112: 'Cn', 113: 'Nh', 114: 'Fl', 115: 'Mc', 116: 'Lv', 117: 'Ts', 118: 'Og'
}

In [41]:
def xyz_block_from_node_features(xh: torch.tensor, comment: str="", c2a: dict=atomic_num2sym) -> str:
    num_atoms = xh.shape[0]
    xyz_lines = [str(num_atoms), comment]
    for row in xh:
        position = row[:3].cpu().numpy()
        z = c2a[row[-1].long().item()]
        xyz_lines.append(f"{z}\t" + "\t".join([str(x) for x in position]))
    return "\n".join(xyz_lines)

In [42]:
output_dir_basename = "sample_random_TS-20260706-C6H6_5710-0-woSL-woBL-wow3pBC-StepLR-cutoff_12-bz64-C6H6_filtered_test" # UPDATE
output_dir = os.path.abspath(os.path.join("results/", output_dir_basename))
os.makedirs(output_dir, exist_ok=True)

In [43]:
!ls -haltr oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/ | grep C6H6 | grep bz64

drwxr-xr-x   2 guest50 users 4.0K Aug 22  2025 C6H6_5710-0-woSL-woBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet147859ed46b5
drwxr-xr-x   2 guest50 users 4.0K Aug 22  2025 C6H6_5710-0-wSL-woBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet0c6a5355a394
drwxr-xr-x   2 guest50 users 4.0K Aug 22  2025 C6H6_5710-0-woSL-woBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet369b4311ee93
drwxr-xr-x   2 guest50 users 4.0K Aug 23  2025 C6H6_5710-0-wSL-woBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnetd095e07080e3
drwxr-xr-x   2 guest50 users 4.0K Aug 23  2025 C6H6_5710-0-woSL-wBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet22b35e509597
drwxr-xr-x   2 guest50 users 4.0K Aug 23  2025 C6H6_5710-0-wSL-wBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnetad3824c569fd
drwxr-xr-x   2 guest50 users 4.0K Aug 23  2025 C6H6_5710-0-wSL-wBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet53efbe37f1d9
drwxr-xr-x   2 guest50 users 4.0K Aug 23  2025 C6H6_5710-0-woSL-wBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet5

In [44]:
!ls -haltr oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/C6H6_5710-0-woSL-woBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet147859ed46b5

total 1.6G
-rw-r--r--   1 guest50 users  29K Aug 16  2025 leftnet.py
-rw-r--r--   1 guest50 users 163M Aug 19  2025 ddpm-epoch=1053-val-totloss=474.38.ckpt
-rw-r--r--   1 guest50 users 163M Aug 19  2025 ddpm-epoch=1078-val-totloss=479.92.ckpt
-rw-r--r--   1 guest50 users 163M Aug 19  2025 ddpm-epoch=1411-val-totloss=474.10.ckpt
-rw-r--r--   1 guest50 users 163M Aug 20  2025 ddpm-epoch=1824-val-totloss=483.62.ckpt
-rw-r--r--   1 guest50 users 163M Aug 20  2025 ddpm-epoch=1932-val-totloss=485.03.ckpt
-rw-r--r--   1 guest50 users 163M Aug 21  2025 ddpm-epoch=2184-val-totloss=483.80.ckpt
-rw-r--r--   1 guest50 users 163M Aug 21  2025 ddpm-epoch=2368-val-totloss=481.17.ckpt
-rw-r--r--   1 guest50 users 163M Aug 21  2025 ddpm-epoch=2370-val-totloss=480.35.ckpt
-rw-r--r--   1 guest50 users 163M Aug 22  2025 ddpm-epoch=2501-val-totloss=483.48.ckpt
-rw-r--r--   1 guest50 users 163M Aug 22  2025 ddpm-epoch=2512-val-totloss=482.90.ckpt
drwxr-xr-x   2 guest50 users 4.0K Aug 22  2025 .
drwxr-xr-x 1

In [45]:
#checkpoints_to_try = ["./oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/leftnet-SCAN-6-w_selfoops-lr2.5e-4-rcmconly_passerini-03516f3022c5/ddpm-epoch=1899-val-totloss=736.23.ckpt"]

In [46]:
from collections import defaultdict
per_checkpoint_ts_rmsds = defaultdict(list)

In [47]:
from parse import parse

In [48]:
import os

basedir = os.path.abspath("oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN")
# UPDATE :
run_names = ["C6H6_5710-0-woSL-wBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet22b35e509597", 
             "C6H6_5710-0-wSL-woBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet0c6a5355a394", 
             "C6H6_5710-0-wSL-woBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnetd095e07080e3", 
             "C6H6_5710-0-wSL-wBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnetad3824c569fd", 
             "C6H6_5710-0-wSL-wBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet53efbe37f1d9",
             "C6H6_5710-0-woSL-wBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet572fb8db0211",
             "C6H6_5710-0-woSL-woBL-w3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet369b4311ee93",
             "C6H6_5710-0-woSL-woBL-wo3pBC-StepLR-cutoff_12-bz64-rep0-SCAN-leftnet147859ed46b5",
             "C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0",
             "combined_C6H6_570-filtered_and_Ts1x-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet058abd800eca"
            ]

selected_checkpoint_paths = {}
self_loops=True
lr = "5e-4"
for run_name in run_names:
    print(run_name)
    runpath = os.path.join(basedir, run_name)
    #parsed = parse("9{self_loops}-{lr}e-4-{etc}", run_name)
    #self_loops = parsed["self_loops"]+"_self_loops"
    name_parts = run_name.split("-")
    lrs = None
    if "StepLR" in name_parts:
        lrs = "StepLR"
    elif "CosAnnl" in name_parts:
        lrs = "CosineAnnealing"
    #lr = parsed["lr"]+"e-4"
 
    # Let's select only the latest saved checkpoint and those among the top-k older checkpoints that have better val-totloss.
    checkpoints = []
    for file in os.listdir(runpath):
         if file.endswith(".ckpt"):
             ckpt_path = os.path.join(runpath, file)
             mod_time = os.path.getmtime(ckpt_path)
             checkpoints.append(tuple([file, mod_time]))
    checkpoints.sort(key=lambda x: x[1], reverse=True) # Most recent first.
    
    selected_checkpoints = []
    seen_val_totloss = set()
    for ckpt, _ in checkpoints:
        parsed2 = parse("ddpm-epoch={epoch}-val-totloss={val-totloss}.ckpt", ckpt)
        val_totloss = float(parsed2["val-totloss"])
        epoch = int(parsed2["epoch"])
        if all(val_totloss < value for value in seen_val_totloss):
            seen_val_totloss.add(val_totloss)
            selected_checkpoints.append(ckpt)
            ckpt_path = os.path.join(runpath, ckpt)
            selected_checkpoint_paths[ckpt_path] = {"self_loops":self_loops, "lr":lr, "lr_scheduler":lrs, "epoch":epoch, "val-totloss":val_totloss} # , "rmsds":{}
    print(list(reversed(selected_checkpoints)))
    print(len(selected_checkpoints))
    print("")

C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0
['ddpm-epoch=2932-val-totloss=457.68.ckpt', 'ddpm-epoch=2947-val-totloss=465.87.ckpt']
2

/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/combined_C6H6_570-filtered_and_Ts1x-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet058abd800eca
['ddpm-epoch=2932-val-totloss=466.18.ckpt', 'ddpm-epoch=2947-val-totloss=470.70.ckpt']
2



In [49]:
all([x==1 for x in dataset.raw_dataset["single_fragment"]])

True

In [50]:
corr_plot_data = {"val-totloss":[], "rmsd":[], "rxn_id":[], "epoch":[], "checkpoint":[], "num_atoms":[]}
per_rxn_rmsds = defaultdict(list)

In [ ]:
selected_representation_triples = {}
rs_rmsdsx = []
ts_rmsdsx = []
ps_rmsdsx = []

selected_indices = selected_test_indices # valid, not train

for i in range(len(dataset)): 
    representations, res = next(itl)
    if i in selected_indices: #random_indices:
        xyz_blocks = []
        print(i)
        test_idx = test_pkl["use_ind"][i]
        print(test_idx)
        rxn_id = test_pkl["reactant"]["rxn"][test_idx]
        print(rxn_id)
        with open(f'{output_dir}/rmsds_per_model.txt', 'a', encoding='utf-8') as f:
                f.write(f"{rxn_id}\n")
        n_samples = representations[0]["size"].size(0)
        fragments_nodes = [
            repre["size"] for repre in representations
        ]
        conditions = torch.tensor([[0] for _ in range(n_samples)], device=device)
        # skipping permutation of indices in reactant state
        xh_fixed = [
            torch.cat(
                [repre[feature_type] for feature_type in FEATURE_MAPPING],
                dim=1,
            )
            for repre in representations
        ]
        print(xh_fixed[2].shape[0])
        print(test_pkl["reactant"]['num_atoms'][test_idx])
        assert xh_fixed[2].shape[0] == test_pkl["reactant"]['num_atoms'][test_idx]
        #ground_truth_ts = xh_fixed[1]

        if output_dir is not None:            
            # Now wrap up XYZs into an output file with informative comments.
            file_name = f"{rxn_id}.xyz"
            # Reactant state, two versions
            rs_ref_xyz = xyz_block_from_node_features(xh_fixed[0], comment=f"True/calculated reference reactant state.")
            xyz_blocks.append(rs_ref_xyz)
            #rs_rec_xyz = xyz_block_from_node_features(out_samples[0][0], comment=f"Reconstructed reactant state. RMSD: {str(round(rs_rmsds[0],6))} Å.")
            #xyz_blocks.append(rs_rec_xyz)
            
            # Transition state, two versions
            ts_ref_xyz = xyz_block_from_node_features(xh_fixed[1], comment=f"True/calculated reference transition state.")
            xyz_blocks.append(ts_ref_xyz)
            #ts_gen_xyz = xyz_block_from_node_features(out_samples[0][1], comment=f"Generated/inpainted transition state. RMSD: {str(round(ts_rmsds[0],6))} Å.")
            #xyz_blocks.append(ts_gen_xyz)
    
            # Product state, two versions
            ps_ref_xyz = xyz_block_from_node_features(xh_fixed[2], comment=f"True/calculated reference product state.")
            xyz_blocks.append(ps_ref_xyz)
            #ps_rec_xyz = xyz_block_from_node_feattotures(out_samples[0][2], comment=f"Reconstructed product state. RMSD: {str(round(ps_rmsds[0],6))} Å.")
            #xyz_blocks.append(ps_rec_xyz)

        for checkpoint_path, ckpt_vals in selected_checkpoint_paths.items():
            checkpoint_name = f"{os.path.dirname(checkpoint_path)}/{os.path.basename(checkpoint_path)}" 
            ddpm_trainer = prep_ddpm_trainer(checkpoint_path, device=device)
            out_samples, out_masks = ddpm_trainer.ddpm.inpaint(
                n_samples=n_samples,
                fragments_nodes=fragments_nodes,
                conditions=conditions,
                return_frames=1,
                resamplings=5,
                jump_length=5,
                timesteps=None,
                xh_fixed=xh_fixed,
                frag_fixed=[0, 2],
            )
            
            # # reactant state (ts_..):
            # rs_rmsds = batch_rmsd(
            #     fragments_nodes, 
            #     out_samples[0],
            #     xh_fixed,
            #     idx=0,
            # )
            # print(rs_rmsds)
            # rs_rmsdsx.append(rs_rmsds[0])
    
            # transition state (ts_..):
            ts_rmsds = batch_rmsd(
                fragments_nodes, 
                out_samples[0],
                xh_fixed,
                idx=1,
            )
            print(f"{checkpoint_name}: {ts_rmsds[0]}")
            with open(f'{output_dir}/rmsds_per_model.txt', 'a', encoding='utf-8') as f:
                f.write(f"{checkpoint_name}: {ts_rmsds[0]}\n")
            per_checkpoint_ts_rmsds[checkpoint_name].append(ts_rmsds[0])
            corr_plot_data["val-totloss"].append(ckpt_vals["val-totloss"])
            corr_plot_data["rmsd"].append(ts_rmsds[0])
            corr_plot_data["rxn_id"].append(rxn_id)
            corr_plot_data["checkpoint"].append(checkpoint_name)
            corr_plot_data["num_atoms"].append(test_pkl["reactant"]['num_atoms'][test_idx])
            corr_plot_data["epoch"].append(ckpt_vals["epoch"])

            if output_dir is not None:
                ts_gen_xyz = xyz_block_from_node_features(out_samples[0][1], comment=f"Generated/inpainted transition state. RMSD: {str(round(ts_rmsds[0],6))} Å. By model {checkpoint_name}.")
                xyz_blocks.append(ts_gen_xyz)
        
            # # product state (ps_..):
            # ps_rmsds = batch_rmsd(
            #     fragments_nodes, 
            #     out_samples[0],
            #     xh_fixed,
            #     idx=2,
            # )
            # print(ps_rmsds)
            # ps_rmsdsx.append(ps_rmsds[0])
            #assert len(rmsds) == 1
        print("")
        with open(f'{output_dir}/rmsds_per_model.txt', 'a', encoding='utf-8') as f:
            f.write("\n")

        if output_dir is not None:
            with open(os.path.join(output_dir, file_name), "w") as f_out:
                f_out.write("\n".join(xyz_blocks))
            
        if i == selected_indices[-1]: # no need to keep iterating.
            break

1
64
C6H6_5710-TS142 CON(17,23)
12
12
/misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0/ddpm-epoch=2947-val-totloss=465.87.ckpt: 0.3853675459072837
/misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0/ddpm-epoch=2932-val-totloss=457.68.ckpt: 0.5344033638748634


In [ ]:
selected_rxns_dict

In [ ]:
1-3/13

In [ ]:
#select_rxns
pprint(dict(sorted(selected_rxns_dict.items())))

In [ ]:
# Report/analysis of RMSDs seen:

#rs_mean = round(np.mean(rs_rmsdsx), 6)
#rs_std = round(np.std(rs_rmsdsx), 6)
#print(f"Reactant state reconstruction RMSD was mean ± std.dev.: \t{rs_mean} \t± {rs_std} Å.")
ts_mean = round(np.mean(ts_rmsds), 6)
ts_std = round(np.std(ts_rmsds), 6)
print(f"Transition state inpainting RMSD was mean ± std.dev.: \t\t{ts_mean} \t± {ts_std} Å.")
#ps_mean = round(np.mean(ps_rmsdsx), 6)
#ps_std = round(np.std(ps_rmsdsx), 6)
#print(f"Product state reconstruction RMSD was mean ± std.dev.: \t\t{ps_mean} \t± {ps_std} Å.")

Tried checkpoints, fraction of RMSD==1.0, and mean RMSD:

`checkpoint_path="./oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/leftnet-SCAN-6-w_selfoops-lr2.5e-4-rcmconly_passerini-03516f3022c5/...`

`ddpm-epoch=999-val-totloss=770.24.ckpt`: ...    

`ddpm-epoch=1799-val-totloss=787.66.ckpt`: 12/20, 0.819974 	± 0.273995 Å.


In [ ]:
for checkpoint_name, ts_rmsds in per_checkpoint_ts_rmsds.items():
    print(checkpoint_name)
    ts_mean = round(np.mean(ts_rmsds), 6)
    ts_std = round(np.std(ts_rmsds), 6)
    failures = len([x for x in ts_rmsds if x == 1.0])
    print(f"    Transition state inpainting RMSD was mean ± std.dev.: \t\t{ts_mean} \t± {ts_std} Å.")
    print(f"    Number of 1.0 RMSD indicating complete failure: {failures} out of {len(ts_rmsds)}.")
    print("")

In [ ]:
output_dir

In [ ]:
import pandas as pd

In [ ]:
for key, val in corr_plot_data.items():
    print(f"{key}: {len(val)}")

In [ ]:
if len(corr_plot_data["epoch"]) == 0:
    fix_epochs = []
    for checkpoint_name in corr_plot_data["checkpoint"]:
        parts = checkpoint_name.split("-")
        for part in parts:
            if part[:6] == "epoch=":
                epoch = int(part[6:])
                fix_epochs.append(epoch)
    corr_plot_data["epoch"] = fix_epochs

In [ ]:
df = pd.DataFrame(data=corr_plot_data)
df

In [ ]:
with open(f'{output_dir}/corr_plot_data.json', 'w', encoding='utf-8') as f:
    json.dump(corr_plot_data, f, indent=4, ensure_ascii=False)

with open(f'{output_dir}/per_checkpoint_ts_rmsds.json', 'w', encoding='utf-8') as f:
    json.dump(per_checkpoint_ts_rmsds, f, indent=4, ensure_ascii=False)

df.to_csv(f'{output_dir}/corr_plot_data.csv', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ax = df.plot(x="val-totloss", y="rmsd", kind="scatter")

fig = ax.get_figure()
fig.savefig(f'{output_dir}/rmsd_vs_val-totloss_plot.png', dpi=300, bbox_inches='tight')

In [ ]:
df.plot(x="epoch", y="val-totloss", kind="scatter")
fig = ax.get_figure()
fig.savefig(f'{output_dir}/val-totloss_vs_epoch_plot.png', dpi=300, bbox_inches='tight')

In [ ]:
df.plot(x="epoch", y="rmsd", kind="scatter")
fig = ax.get_figure()
fig.savefig(f'{output_dir}/rmsd_vs_epoch_plot.png', dpi=300, bbox_inches='tight')

In [ ]:
import ase
from ase.io import read, write

In [ ]:
output_dir

In [ ]:
!ls -haltr /misc/home/guest50/OAReactDiff/results/sample_random_TS-20260703-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_test

In [ ]:
atoms = read(output_dir+'/C6H6_5710-TS18506 CON(1516,1193).xyz', format='xyz',index=":")

In [ ]:
atoms[1]

In [ ]:
from io import StringIO

def atoms_to_xyz_str(atoms: ase.Atoms):
    f = StringIO()
    atoms.write(f, format="xyz")
    return f.getvalue()

In [ ]:
xyz_01 = atoms_to_xyz_str(atoms[1])
print(xyz_01)

In [ ]:
draw_in_3dmol(xyz_01)

In [ ]:
draw_in_3dmol(atoms_to_xyz_str(atoms[-1]))